In [5]:
import pandas as pd
import quantstats as qs 
import yfinance as yf


In [6]:
dados_empresas = pd.read_excel("statusinvest-busca-avancada.xlsx")


In [7]:
##caso queira tickers unicos
##def get_base_ticker(ticker):
##    return ''.join(filter(str.isalpha, ticker))


##def get_priority(ticker):
##    if ticker.endswith('4'):
##        return 3
##    elif ticker.endswith('11'):
##        return 2
##    elif ticker.endswith('3'):
##        return 1
##    else:
##        return 0

##dados_empresas['TICKER'] = dados_empresas['TICKER'].apply(get_base_ticker)

##dados_empresas['Priority'] = dados_empresas['TICKER'].apply(get_priority)

##dados_empresas = dados_empresas.sort_values(by=['TICKER', 'Priority'], ascending=[True, False])

##dados_empresas = dados_empresas.drop_duplicates(subset='TICKER', keep='first')

In [8]:

dados_empresas = dados_empresas[dados_empresas[' VALOR DE MERCADO'] > 1000000]
dados_empresas = dados_empresas[dados_empresas[' LIQUIDEZ MEDIA DIARIA'] > 500000]
dados_empresas = dados_empresas[dados_empresas['MARG. LIQUIDA'] >=0]
dados_empresas = dados_empresas[dados_empresas['MARGEM BRUTA'] >=0]
dados_empresas = dados_empresas[dados_empresas['MARGEM EBIT'] >=0]
dados_empresas = dados_empresas[dados_empresas['CAGR RECEITAS 5 ANOS'] >= 0]
dados_empresas = dados_empresas[dados_empresas['CAGR LUCROS 5 ANOS'] >= 0]
dados_empresas = dados_empresas[dados_empresas['ROE'] >= 0]
dados_empresas = dados_empresas[dados_empresas['LIQ. CORRENTE'] != 0]
dados_empresas = dados_empresas[dados_empresas[' PEG Ratio'] > 0]
dados_empresas = dados_empresas[dados_empresas['P/VP'] <= 2]
dados_empresas.fillna(0, inplace=True)

In [9]:
dados_empresas['ranking_margem_liq'] = dados_empresas['MARG. LIQUIDA'].rank(ascending = False)

dados_empresas['ranking_pl'] = dados_empresas['P/L'].rank(ascending = True)

dados_empresas['ranking_psr'] = dados_empresas['PSR'].rank(ascending = False)

dados_empresas['ranking_roe'] = dados_empresas['ROE'].rank(ascending = False)

dados_empresas['ranking_p_ebit'] = dados_empresas['P/EBIT'].rank(ascending = True)

dados_empresas['ranking_lpa'] = dados_empresas[' LPA'].rank(ascending = False)

dados_empresas['ranking_liq_corr'] = dados_empresas['LIQ. CORRENTE'].rank(ascending = False)

dados_empresas['ranking_mktValue'] = dados_empresas[' VALOR DE MERCADO'].rank(ascending = False)

In [10]:
dados_empresas['ranking_final'] = dados_empresas['ranking_margem_liq'] + dados_empresas['ranking_psr'] + dados_empresas['ranking_roe'] + dados_empresas['ranking_liq_corr'] + dados_empresas['ranking_lpa'] + dados_empresas['ranking_p_ebit'] + dados_empresas['ranking_pl'] + dados_empresas['ranking_mktValue']

dados_empresas['ranking_final'] = dados_empresas['ranking_final'].rank()

In [11]:
dados_empresas.sort_values('ranking_final').head(10)

,TICKER,PRECO,DY,P/L,P/VP,P/ATIVOS,MARGEM BRUTA,MARGEM EBIT,MARG. LIQUIDA,P/EBIT,...,VALOR DE MERCADO,ranking_margem_liq,ranking_pl,ranking_psr,ranking_roe,ranking_p_ebit,ranking_lpa,ranking_liq_corr,ranking_mktValue,ranking_final
464,PRIO3,43.63,0.00,3.45,1.47,0.71,43.99,40.30,72.61,6.22,...,3.908966e+10,4.0,2.0,8.0,1.0,33.0,2.0,9.0,9.0,1.0
328,ISAE4,22.56,10.47,4.16,0.73,0.33,46.33,63.15,42.48,2.80,...,1.709700e+10,6.0,4.0,14.0,26.0,4.0,7.0,5.0,19.0,2.0
544,TAEE11,33.46,8.20,6.83,1.58,0.55,63.45,73.56,42.45,3.94,...,1.150971e+10,7.5,22.0,6.0,9.5,11.0,8.0,22.5,21.5,3.0
505,SAPR11,36.17,6.11,4.67,1.07,0.55,56.39,39.68,34.03,4.00,...,1.091090e+10,10.0,7.0,16.0,12.0,13.0,5.0,26.0,24.0,4.0
337,JHSF3,5.13,7.25,3.28,0.61,0.24,58.77,90.78,62.12,2.24,...,3.495186e+09,5.0,1.0,9.0,19.0,1.0,29.0,18.0,37.0,5.0
509,SBSP3,111.97,3.33,7.48,1.99,0.89,53.87,43.91,26.93,4.59,...,7.649842e+10,14.0,27.0,10.0,4.0,19.0,1.0,44.0,6.0,6.0
546,TAEE4,11.21,8.15,6.87,1.59,0.55,63.45,73.56,42.45,3.96,...,1.150971e+10,7.5,24.0,5.0,9.5,12.0,26.0,22.5,21.5,7.0
507,SAPR4,7.07,6.38,4.55,1.05,0.54,56.39,39.68,34.03,3.90,...,1.091090e+10,10.0,6.0,18.0,12.0,10.0,30.5,26.0,24.0,8.0
506,SAPR3,7.79,5.26,5.02,1.16,0.60,56.39,39.68,34.03,4.31,...,1.091090e+10,10.0,9.0,15.0,12.0,14.0,30.5,26.0,24.0,9.0
200,CYRE3,26.63,4.02,5.97,1.09,0.47,32.60,24.36,20.48,5.02,...,1.020672e+10,19.0,14.0,25.5,20.0,24.0,9.0,7.0,26.0,10.0


# Passo 6: Criar  as carteiras. 

In [12]:
dados_empresas = dados_empresas[dados_empresas['ranking_final'] <= 10]
dados_empresas = dados_empresas["TICKER"]
dados_empresas

200     CYRE3
328     ISAE4
337     JHSF3
464     PRIO3
505    SAPR11
506     SAPR3
507     SAPR4
509     SBSP3
544    TAEE11
546     TAEE4
Name: TICKER, dtype: object

In [20]:
tickers = pd.Series(dados_empresas).tolist()
tickers = [ticker + ".SA" for ticker in tickers]

##transformando os tickers de dataframe pra lista e adicionando o SA para encontrar no yf

inicio = "2011-01-01"
fim = "2025-06-30"
data = yf.download(tickers, start=inicio, end=fim).dropna()
data = data["Close"]
data

C:\Users\adm\AppData\Local\Temp\ipykernel_19572\2700759289.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=inicio, end=fim).dropna()
[*********************100%***********************]  10 of 10 completed


Ticker,CYRE3.SA,ISAE4.SA,JHSF3.SA,PRIO3.SA,SAPR11.SA,SAPR3.SA,SAPR4.SA,SBSP3.SA,TAEE11.SA,TAEE4.SA
Date,,,,,,,,,,
2017-12-26,8.979933,4.294104,1.172884,1.575469,12.234171,2.783570,2.310587,28.218466,10.110988,3.325635
2017-12-27,9.034898,4.281647,1.186288,1.614407,12.238334,2.739735,2.359747,28.366465,10.091819,3.325635
2017-12-28,9.082993,4.385886,1.172884,1.633377,12.473649,2.698091,2.396619,28.226690,10.225990,3.325635
2018-01-02,9.151699,4.418011,1.206395,1.706259,12.622127,2.973677,2.381177,28.448681,10.283501,3.325635
2018-01-03,9.192924,4.328193,1.253310,1.737210,12.513004,3.047795,2.372748,27.938910,10.297874,3.379432
...,...,...,...,...,...,...,...,...,...,...
2025-06-23,25.379999,23.250000,5.209984,43.250000,35.275753,7.019736,7.070738,114.029999,34.259998,11.400000
2025-06-24,25.780001,23.120001,5.249756,41.560001,35.150558,7.155102,6.984158,114.910004,34.299999,11.450000
2025-06-25,25.150000,22.900000,5.190099,41.029999,35.429836,7.309807,6.964917,113.680000,34.459999,11.460000


In [21]:
def retorno (data):
    retorno = data[-1] / data[0] -1
    return retorno 

In [16]:
data["CYRE3.SA"].pct_change()

Date
2017-12-26         NaN
2017-12-27    0.006121
2017-12-28    0.005323
2018-01-02    0.007564
2018-01-03    0.004505
                ...   
2025-06-23    0.004751
2025-06-24    0.015761
2025-06-25   -0.024438
2025-06-26    0.027435
2025-06-27   -0.008514
Name: CYRE3.SA, Length: 1859, dtype: float64